## Load/Prep Data

In [1]:
import sys
!{sys.executable} -m pip install grn-dazzle scanpy scipy scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import scanpy as sc
from scipy.stats import pearsonr
from sklearn.preprocessing import minmax_scale
from dazzle import runDAZZLE, DEFAULT_DAZZLE_CONFIGS
import scipy.sparse as sp

In [3]:
# 1. Load Metadata and Expression Data
print("Loading data...")
metadata = pd.read_csv('Data/CHOOSE-multiome-wt-metadata.csv', index_col=0)

# Pandas can read .csv.gz directly
expr_data = pd.read_csv('Data/CHOOSE-multiome-wt-log-norm.csv.gz', index_col=0) 

# Ensure the expression matrix columns match metadata rows
expr_data = expr_data[metadata.index]

# Convert to an AnnData object for easier slicing by cell type
adata = sc.AnnData(X=expr_data.T)
adata.obs['celltype_jf'] = metadata['celltype_jf']

print(f"Loaded {adata.n_obs} cells and {adata.n_vars} genes.")

Loading data...
Loaded 732 cells and 15685 genes.


## Run DAZZLE for each cell type (going off of GRNBoost2)

In [4]:
import torch
# FORCE CPU MODE TO PREVENT OOM CRASH
torch.cuda.is_available = lambda : False 
device = torch.device("cpu")
# 1. SET THE CELL TYPE AND TARGET TFs
current_ct = 'ctx_ip' 

# TFs of interest for the "ge_in" cell type
target_tfs = [
    'BCL11A', 'TCF4', 'ARID1B', 'ADNP', 'ASH1L', 'CHD8', 
    'DYRK1A', 'POGZ', 'SHANK3', 'SYNGAP1', 'TBR1', 
    'SCN2A', 'GRIN2B', 'FOXP1'
]

print(f"--- Starting: {current_ct} ---")

# 2. Subset and filter
adata_ct = adata[adata.obs['celltype_jf'] == current_ct].copy()
sc.pp.filter_genes(adata_ct, min_cells=1)

# 3. Run DAZZLE
expr_array = adata_ct.X.toarray() if sp.issparse(adata_ct.X) else adata_ct.X
model, adjs = runDAZZLE(expr_array, DEFAULT_DAZZLE_CONFIGS)

# 4. Optimized Slicing (The fix that prevents the crash)
# This uses the 'target_tfs' list defined above to extract only relevant rows
adj_matrix = model.get_adj()
gene_names = np.array(adata_ct.var_names)
present_tfs = [gene for gene in target_tfs if gene in gene_names]
tf_indices = [np.where(gene_names == gene)[0][0] for gene in present_tfs]



reduced_adj = adj_matrix[tf_indices, :] 

# 5. Format results
adj_df = pd.DataFrame(reduced_adj, index=present_tfs, columns=gene_names)
dazzle_edges = adj_df.reset_index().melt(id_vars='index', var_name='target', value_name='importance')
dazzle_edges.rename(columns={'index': 'TF'}, inplace=True)
dazzle_edges['cell.type'] = current_ct

# 6. SAVE TO DISK IMMEDIATELY
filename = f"dazzle_results_{current_ct}.csv"
dazzle_edges.to_csv(filename, index=False)

print(f"Successfully finished {current_ct} and saved to {filename}")

--- Starting: ctx_ip ---
Cuda is not available for torch. Proceed on CPU instead.


100%|██████████| 120/120 [2:35:48<00:00, 77.90s/it] 


Successfully finished ctx_ip and saved to dazzle_results_ctx_ip.csv


# Post process - calculating the Pearson correlation between the TF and the target gene.

In [5]:
def calculate_mor_and_format(predictions_df, adata_obj):
    """
    Translates the R post-processing logic to Python.
    Calculates Pearson correlation to determine the sign (+ or -) of the edge,
    then scales the final score to [-1, 1].
    """
    final_results = []
    
    # Group by cell type so we calculate correlations within the correct cell population
    for cell_type, group in predictions_df.groupby('cell.type'):
        # Get the expression matrix for this cell type
        ct_expr = adata_obj[adata_obj.obs['celltype_jf'] == cell_type].to_df()
        
        for _, row in group.iterrows():
            tf = row['TF']
            target = row['target']
            importance = row['importance']
            
            # Skip if TF or target isn't in our expression matrix
            if tf not in ct_expr.columns or target not in ct_expr.columns:
                continue
                
            # 1. Calculate Pearson correlation between TF and target expression
            # This gives us the direction of regulation (positive = induce, negative = repress)
            tf_expr = ct_expr[tf].values
            target_expr = ct_expr[target].values
            
            # Note: if a gene has zero variance (e.g., all 0s), correlation will be NaN
            if np.std(tf_expr) == 0 or np.std(target_expr) == 0:
                corr = 0
            else:
                corr, _ = pearsonr(tf_expr, target_expr)
            
            # 2. Multiply DAZZLE's non-negative importance by the sign of the correlation
            raw_score = importance * np.sign(corr)
            
            final_results.append({
                'TF': tf,
                'target': target,
                'cell.type': cell_type,
                'raw_score': raw_score,
                'importance': importance # Keep for p-value calculation if needed
            })
            
    res_df = pd.DataFrame(final_results)
    
    # 3. Scale the scores so the maximum absolute value is 1 (matches the R script)
    # all.res[,"score"] <- all.res[,"score"] / max(abs(all.res[,"score"]))
    max_abs_score = res_df['raw_score'].abs().max()
    res_df['score'] = res_df['raw_score'] / max_abs_score
    
    return res_df[['TF', 'cell.type', 'target', 'score']]

# Run the post-processing
final_submission = calculate_mor_and_format(all_predictions, adata)

# Save your final predictions!
final_submission.to_csv('dazzle_predictions_submission.csv', index=False)
print("Ready to submit!")

NameError: name 'all_predictions' is not defined